In [1]:
            import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt

In [3]:
from scipy.stats import beta

In [4]:
import pandas as pd


In [5]:
import numpy as np

In [6]:
import itertools

In [7]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [8]:
import scipy.io

control = scipy.io.loadmat('../data/cps_controls.mat')

In [9]:
import seaborn as sns

In [10]:
def process_data(file):
    
    treated = scipy.io.loadmat(file)
    
    data = pd.DataFrame(np.squeeze(np.array([treated["t"], treated["age"], treated["education"], 
                                  treated["black"], treated["hispanic"], treated["married"],
                                  treated["nodegree"], treated["re74"], treated["re75"], treated["re78"],
                                  treated["u74"], treated["u75"],
                                 ]), axis = -1)).transpose()
    
    data.columns = ["t", "age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75", "re78", "u74", "u75"]
    
    return data

In [11]:
df_t = process_data("../data/nswre74_treated.mat")
df_c = process_data("../data/cps_controls.mat")

In [12]:
df = pd.concat([df_t, df_c])

In [13]:
df["t"] = df["t"].apply(int)

In [14]:
df["re74"] = df["re74"] / 1000
df["re75"] = df["re75"] / 1000
df["re78"] = df["re78"] / 1000

a)

In [15]:
df_diff =df.groupby(["t"]).agg(["mean", "std", "count"]).stack([0,1]).unstack([- 1, "t"]).sort_index(axis = 1)

In [16]:
df_diff["std_norm"] = ((df_diff["std"][1]**2 + df_diff["std"][0]**2)/2)**0.5
df_diff["std_diff"] = ((df_diff["std"][1]**2/df_diff["count"][1] + df_diff["std"][0]**2/df_diff["count"][0])**0.5)

In [17]:
df_diff["nd"] = (df_diff["mean"][1] - df_diff["mean"][0] ) / df_diff["std_norm"]
df_diff["t"] = (df_diff["mean"][1] - df_diff["mean"][0] ) / df_diff["std_diff"]

In [18]:
df_diff = df_diff.drop(["re78"])
df_diff

count              mean                   std            \
t                0      1          0          1          0         1   
age        15992.0  185.0  33.225238  25.816216  11.045216  7.155019   
black      15992.0  185.0   0.073537   0.843243   0.261024  0.364558   
education  15992.0  185.0  12.027514  10.345946   2.870846  2.010650   
hispanic   15992.0  185.0   0.072036   0.059459   0.258556  0.237124   
married    15992.0  185.0   0.711731   0.189189   0.452971  0.392722   
nodegree   15992.0  185.0   0.295835   0.708108   0.456432  0.455867   
re74       15992.0  185.0  14.016800   2.095574   9.569796  4.886620   
re75       15992.0  185.0  13.650803   1.532055   9.270403  3.219251   
u74        15992.0  185.0   0.119622   0.708108   0.324530  0.455867   
u75        15992.0  185.0   0.109305   0.600000   0.312031  0.491227   

           std_norm  std_diff        nd          t  
t                                                   
age        9.305673  0.533249 -0.796183 -13.894110  
black      0.317046  0.026882  2.427747  28.632580  
education  2.478353  0.149559 -0.678502 -11.243512  
hispanic   0.248072  0.017553 -0.050697  -0.716482  
married    0.423918  0.029095 -1.232648 -17.959964  
nodegree   0.456149  0.033710  0.903811  12.230069  
re74       7.598028  0.367155 -1.568990 -32.469208  
re75       6.939162  0.247777 -1.746428 -48.909959  
u74        0.395685  0.033614  1.487257  17.507121  
u75        0.411502  0.036200  1.192450  13.555140

b)

In [19]:
variables_check = ["age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75", "u74", "u75"]

In [20]:
df.columns

Index(['t', 'age', 'education', 'black', 'hispanic', 'married', 'nodegree',
       're74', 're75', 're78', 'u74', 'u75'],
      dtype='object')

In [21]:
model  = smf.probit("t ~ 1 + age + education + black + hispanic + married + nodegree + re74 + re75 + u74 + u75", data = df).fit()

Optimization terminated successfully.
         Current function value: 0.029675
         Iterations 11


In [22]:
df["e_X"] = model.predict()

In [23]:
sns.kdeplot(df[df.t == 0].e_X)
sns.kdeplot(df[df.t == 1].e_X)
plt.savefig("propensity.png")
plt.close()

c)

Weigths for the ATE

In [24]:
m = smf.probit("t ~ 1 + age + education + black + hispanic + married + nodegree + re74 + re75 + u74 + u75", data = df)

In [25]:
m.fit()
print("X2")

Optimization terminated successfully.
         Current function value: 0.029675
         Iterations 11
X2


In [26]:
def get_estimate_te_probit(df):
    # Estimate propensity
    model  = smf.probit("t ~ 1 + age + education + black + hispanic + married + nodegree + re74 + re75 + u74 + u75", data = df).fit(disp = 0)
    df["e_X"] = model.predict()
    
    N = df.shape[0]
    N_t = df["t"].sum()

    df["w_ate_t"] = (1 / df.e_X)
    df["w_ate_c"] = (1 / (1 - df.e_X))
    
    df["w_ate"] = (df.t / df.e_X - (1 - df.t) / (1 - df.e_X))

    df["w_att_t"] = 1
    df["w_att_c"] =  df.e_X / (1 - df.e_X)
    
    df["w_att"] = df.t - (1 - df.t) * df.e_X / (1 - df.e_X)
    
    ate = (df["w_ate"] * df["re78"]).sum()/N
    att = (df["w_att"] * df["re78"]).sum()/N_t

    return ate, att, df

In [27]:
ate, att, df_probit = get_estimate_te_probit(df)

In [28]:
ate

-5.542757942680477

In [29]:
att

1.4008460929181834

In [30]:
normalized_weights = df_probit[df_probit.t == 0]["w_att_c"].pipe(lambda y: y - y.mean() + 1)

In [31]:
normalized_weights.describe()

count    15992.000000
mean         1.000000
std          0.073382
min          0.988852
25%          0.988854
50%          0.988875
75%          0.989677
max          2.264426
Name: w_att_c, dtype: float64

In [32]:
sns.kdeplot(normalized_weights)
plt.savefig("norm_weights.png")
plt.close()

In [33]:
df_probit.describe()

,t,age,education,black,hispanic,married,nodegree,re74,re75,re78,u74,u75,e_X,w_ate_t,w_ate_c,w_ate,w_att_t,w_att_c,w_att
count,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,1.617700e+04,1.617700e+04,16177.000000,16177.000000,16177.0,1.617700e+04,16177.000000
mean,0.011436,33.140508,12.008283,0.082339,0.071892,0.705755,0.300550,13.880469,13.512214,14.749482,0.126352,0.114916,1.143629e-02,1.069749e+06,1.017214,0.122799,1.0,1.721415e-02,0.000416
std,0.106329,11.036508,2.868005,0.274889,0.258317,0.455717,0.458511,9.613115,9.313207,9.670996,0.332256,0.318931,5.532375e-02,3.691512e+06,0.103219,65.898601,0.0,1.032190e-01,0.129934
min,0.000000,16.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.802004e-09,1.783961e+00,1.000000,-2.275573,1.0,4.802005e-09,-1.275573
25%,0.000000,24.000000,11.000000,0.000000,0.000000,0.000000,0.000000,4.075275,4.103419,5.492710,0.000000,0.000000,1.232289e-06,1.034559e+03,1.000001,-1.000777,1.0,1.232291e-06,-0.000777
50%,0.000000,31.000000,12.000000,0.000000,0.000000,1.000000,0.000000,14.892390,14.374500,16.240220,0.000000,0.000000,2.458182e-05,4.068048e+04,1.000025,-1.000021,1.0,2.458242e-05,-0.000021
75%,0.000000,42.000000,13.000000,0.000000,0.000000,1.000000,1.000000,23.491610,22.830190,25.564670,0.000000,0.000000,9.665958e-04,8.114979e+05,1.000968,-1.000001,1.0,9.675311e-04,-0.000001
max,1.000000,55.000000,18.000000,1.000000,1.000000,1.000000,1.000000,35.040070,25.243550,60.307930,1.000000,1.000000,5.605503e-01,2.082464e+08,2.275573,6822.326431,1.0,1.275573e+00,1.000000


In [34]:
df_probit.loc[normalized_weights.nlargest(5).index]

,t,age,education,black,hispanic,married,nodegree,re74,re75,re78,u74,u75,e_X,w_ate_t,w_ate_c,w_ate,w_att_t,w_att_c,w_att
9384,0,16.0,10.0,1.0,0.0,0.0,1.0,0.0,0.0,2.880089,1.0,1.0,0.560550,1.783961,2.275573,-2.275573,1,1.275573,-1.275573
11267,0,16.0,10.0,1.0,0.0,0.0,1.0,0.0,0.0,1.730418,1.0,1.0,0.560550,1.783961,2.275573,-2.275573,1,1.275573,-1.275573
13971,0,16.0,10.0,1.0,0.0,0.0,1.0,0.0,0.0,14.669390,1.0,1.0,0.560550,1.783961,2.275573,-2.275573,1,1.275573,-1.275573
15572,0,17.0,11.0,1.0,0.0,0.0,1.0,0.0,0.0,7.673834,1.0,1.0,0.558672,1.789958,2.265889,-2.265889,1,1.265889,-1.265889
887,0,16.0,9.0,1.0,0.0,0.0,1.0,0.0,0.0,0.000000,1.0,1.0,0.558379,1.790898,2.264386,-2.264386,1,1.264386,-1.264386


In [35]:
n_r = 1000

In [36]:

ate_boot_p= []
att_boot_p= []

for _ in range(n_r):
    ate, att, _ = get_estimate_te_probit(df.sample(df.shape[0], replace = True))
    ate_boot_p.append(ate)
    att_boot_p.append(att)

sns.kdeplot(ate_boot_p)

In [37]:
sns.kdeplot(att_boot_p)
plt.savefig("att_boot_p.png")
plt.close()

In [38]:
np.std(ate_boot_p)

7.02399745428091

In [39]:
np.std(att_boot_p)

0.6728596333299252

d)

In [40]:
def get_estimate_te_logit(df):
    # Estimate propensity
    model  = smf.logit("t ~ 1 + age + education + black + hispanic + married + nodegree + re74 + re75 + u74 + u75", data = df).fit(disp = 0)
    df["e_X"] = model.predict()
    
    N = df.shape[0]
    N_t = df["t"].sum()

    df["w_ate_t"] = (1 / df.e_X)
    df["w_ate_c"] = (1 / (1 - df.e_X))
    
    df["w_ate"] = (df.t / df.e_X - (1 - df.t) / (1 - df.e_X))

    df["w_att_t"] = 1
    df["w_att_c"] =  df.e_X / (1 - df.e_X)
    
    df["w_att"] = df.t - (1 - df.t) * df.e_X / (1 - df.e_X)
    
    ate = (df["w_ate"] * df["re78"]).sum()/N
    att = (df["w_att"] * df["re78"]).sum()/N_t

    return ate, att, df

In [41]:
ate, att, df_logit = get_estimate_te_logit(df)

In [42]:
ate

-10.548411274633189

In [43]:
att

1.293004047686427

In [44]:
normalized_weights = df_logit[df_logit.t == 0]["w_att_c"].pipe(lambda y: y - y.mean() + 1)

In [45]:
normalized_weights.describe()

count    15992.000000
mean         1.000000
std          0.091831
min          0.988241
25%          0.988284
50%          0.988436
75%          0.989814
max          2.746941
Name: w_att_c, dtype: float64

In [46]:
sns.kdeplot(normalized_weights)
plt.savefig("norm_weights_l.png")
plt.close()

In [47]:
df_logit.describe()

,t,age,education,black,hispanic,married,nodegree,re74,re75,re78,u74,u75,e_X,w_ate_t,w_ate_c,w_ate,w_att_t,w_att_c,w_att
count,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.000000,16177.0,16177.000000,16177.000000
mean,0.011436,33.140508,12.008283,0.082339,0.071892,0.705755,0.300550,13.880469,13.512214,14.749482,0.126352,0.114916,0.011436,12990.417550,1.019290,-0.462858,1.0,0.019290,-0.000193
std,0.106329,11.036508,2.868005,0.274889,0.258317,0.455717,0.458511,9.613115,9.313207,9.670996,0.332256,0.318931,0.059658,16831.657285,0.132098,21.302956,0.0,0.132098,0.141102
min,0.000000,16.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000005,1.568600,1.000005,-2.758705,1.0,0.000005,-1.758705
25%,0.000000,24.000000,11.000000,0.000000,0.000000,0.000000,0.000000,4.075275,4.103419,5.492710,0.000000,0.000000,0.000048,573.506456,1.000048,-1.001528,1.0,0.000048,-0.001528
50%,0.000000,31.000000,12.000000,0.000000,0.000000,1.000000,0.000000,14.892390,14.374500,16.240220,0.000000,0.000000,0.000208,4799.037779,1.000208,-1.000192,1.0,0.000208,-0.000192
75%,0.000000,42.000000,13.000000,0.000000,0.000000,1.000000,1.000000,23.491610,22.830190,25.564670,0.000000,0.000000,0.001744,20698.725374,1.001747,-1.000046,1.0,0.001747,-0.000046
max,1.000000,55.000000,18.000000,1.000000,1.000000,1.000000,1.000000,35.040070,25.243550,60.307930,1.000000,1.000000,0.637511,212574.981318,2.758705,1953.595128,1.0,1.758705,1.000000


In [48]:
df_logit.loc[normalized_weights.nlargest(5).index]

,t,age,education,black,hispanic,married,nodegree,re74,re75,re78,u74,u75,e_X,w_ate_t,w_ate_c,w_ate,w_att_t,w_att_c,w_att
15572,0,17.0,11.0,1.0,0.0,0.0,1.0,0.0,0.0,7.673834,1.0,1.0,0.637511,1.568600,2.758705,-2.758705,1,1.758705,-1.758705
9384,0,16.0,10.0,1.0,0.0,0.0,1.0,0.0,0.0,2.880089,1.0,1.0,0.637145,1.569501,2.755924,-2.755924,1,1.755924,-1.755924
11267,0,16.0,10.0,1.0,0.0,0.0,1.0,0.0,0.0,1.730418,1.0,1.0,0.637145,1.569501,2.755924,-2.755924,1,1.755924,-1.755924
13971,0,16.0,10.0,1.0,0.0,0.0,1.0,0.0,0.0,14.669390,1.0,1.0,0.637145,1.569501,2.755924,-2.755924,1,1.755924,-1.755924
2596,0,18.0,11.0,1.0,0.0,0.0,1.0,0.0,0.0,0.000000,1.0,1.0,0.633364,1.578872,2.727498,-2.727498,1,1.727498,-1.727498


In [49]:
ate_boot_l= []
att_boot_l= []

for _ in range(n_r):
    ate, att, _ = get_estimate_te_logit(df.sample(df.shape[0], replace = True))
    ate_boot_l.append(ate)
    att_boot_l.append(att)

sns.kdeplot(ate_boot_l)

plt.close()

In [50]:
sns.kdeplot(att_boot_l)
plt.savefig("att_boot_l.png")
plt.close()

In [51]:
np.std(ate_boot_l)

1.604183677798524

In [52]:
np.std(att_boot_l)

0.6584065058449428

e)

In [53]:
quantiles = list(df_probit[df_probit.t == 1]["e_X"].quantile([0.2, 0.4, 0.6, 0.8]).values)

In [54]:
quantiles =  [-np.inf] + quantiles + [np.inf]

In [55]:
df_probit["strata"] = pd.cut(df_probit["e_X"], quantiles, include_lowest=True)

In [56]:
df_probit.groupby(["t", "strata"])["e_X"].count().unstack()

/tmp/ipykernel_63499/773433147.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_probit.groupby(["t", "strata"])["e_X"].count().unstack()


strata,"(-inf, 0.0847]","(0.0847, 0.221]","(0.221, 0.391]","(0.391, 0.595]","(0.595, inf]"
t,,,,,
0,15640,228,57,41,26
1,37,37,37,37,37


f)

In [57]:
df_diff_strata = df_probit.groupby(["t", "strata"])[variables_check].agg(["count", "mean", "std"])

/tmp/ipykernel_63499/1733743840.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_diff_strata = df_probit.groupby(["t", "strata"])[variables_check].agg(["count", "mean", "std"])


In [58]:
df_diff_strata = df_diff_strata.stack([0,1]).unstack([-1, 0 ])

In [59]:
df_diff_strata["std_norm"] = ((df_diff_strata["std"][1]**2 + df_diff_strata["std"][0]**2)/2)**0.5
df_diff_strata["std_diff"] = ((df_diff_strata["std"][1]**2/df_diff_strata["count"][1] + df_diff_strata["std"][0]**2/df_diff_strata["count"][0])**0.5)

In [60]:
df_diff_strata["nd"] = (df_diff_strata["mean"][1] - df_diff_strata["mean"][0] ) / df_diff_strata["std_norm"]
df_diff_strata["t"] = (df_diff_strata["mean"][1] - df_diff_strata["mean"][0] ) / df_diff_strata["std_diff"]

In [61]:
df_diff[["nd", "t"]]

,nd,t
t,,
age,-0.796183,-13.894110
black,2.427747,28.632580
education,-0.678502,-11.243512
hispanic,-0.050697,-0.716482
married,-1.232648,-17.959964
nodegree,0.903811,12.230069
re74,-1.568990,-32.469208
re75,-1.746428,-48.909959
u74,1.487257,17.507121


In [62]:
df_diff_strata["nd"].unstack()

,age,black,education,hispanic,married,nodegree,re74,re75,u74,u75
strata,,,,,,,,,,
"(-inf, 0.0847]",-0.834958,0.722733,-0.497498,0.352857,-0.670748,0.524810,-1.100325,-1.261007,0.771917,0.504634
"(0.0847, 0.221]",-0.060720,0.181133,-0.295111,-0.181133,-0.216911,0.338794,0.245570,0.413035,-0.336151,-0.498207
"(0.221, 0.391]",-0.461552,NaN,0.307899,NaN,0.093126,-0.111874,-0.060437,0.139844,0.360755,-0.055743
"(0.391, 0.595]",0.359669,NaN,-0.022887,NaN,NaN,-0.115005,NaN,-0.322637,NaN,0.802732
"(0.595, inf]",1.010489,NaN,0.093550,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
df_diff_strata["nd"].unstack().fillna(0).mean()

age          0.002586
black        0.180773
education   -0.082809
hispanic     0.034345
married     -0.158907
nodegree     0.127345
re74        -0.183038
re75        -0.206153
u74          0.159304
u75          0.150683
dtype: float64

g)

In [64]:
df_probit.columns

Index(['t', 'age', 'education', 'black', 'hispanic', 'married', 'nodegree',
       're74', 're75', 're78', 'u74', 'u75', 'e_X', 'w_ate_t', 'w_ate_c',
       'w_ate', 'w_att_t', 'w_att_c', 'w_att', 'strata'],
      dtype='object')

In [65]:
strata_list = df_probit["strata"].sort_values().unique()

In [66]:

list_estimates = [*map(lambda strata: smf.ols("re78 ~ 1 + t + age + education + black + hispanic + married + nodegree + re74 + re75 + u74 + u75", data = df_probit[df_probit.strata == strata] ).fit().params["t"], strata_list)]

In [67]:
df_probit[df_probit.t == 1]["strata"].value_counts()

strata
(-inf, 0.0847]     37
(0.0847, 0.221]    37
(0.221, 0.391]     37
(0.391, 0.595]     37
(0.595, inf]       37
Name: count, dtype: int64

In [68]:
np.mean(list_estimates)

1.8445073423171465

h)

In [69]:
strata_list = df_probit["strata"].sort_values().unique()

In [70]:
strata = strata_list[0]

In [71]:
def get_estimates_bootstrap(df_probit):
    att_per_strata = []
    for strata in strata_list:
        df_str = df_probit[df_probit.strata == strata]
        N = df_str.shape[0]
        df_str = df_str.sample(N, replace = True)
        att_per_strata.append(smf.ols("re78 ~ 1 + t + age + education + black + hispanic + married + nodegree + re74 + re75 + u74 + u75", data = df_str).fit().params["t"])

    return att_per_strata

In [72]:
results_bootstrap = []
for _ in range(n_r):
    results_bootstrap.append(get_estimates_bootstrap(df_probit))

In [73]:
att_bootstrap = pd.DataFrame(results_bootstrap).mean(axis = 1)

In [74]:
sns.kdeplot(att_bootstrap)
plt.savefig("att_boot_d.png")
plt.close()

In [75]:
np.std(att_bootstrap)

0.8413313032792257